In [1]:
import pandas as pd


In [2]:
df = pd.read_json("products.jsonl", lines = True)

In [3]:
df

,name,category,description,ingredients,price,rating,image_path
0,Cappuccino,Coffee,A rich and creamy cappuccino made with freshly...,"[Espresso, Steamed Milk, Milk Foam]",4.50,4.7,cappuccino.jpg
1,Jumbo Savory Scone,Bakery,"Deliciously flaky and buttery, this jumbo savo...","[Flour, Butter, Cheese, Herbs, Baking Powder, ...",3.25,4.3,SavoryScone.webp
2,Latte,Coffee,"Smooth and creamy, our latte combines rich esp...","[Espresso, Steamed Milk, Milk Foam]",4.75,4.8,Latte.jpg
3,Chocolate Chip Biscotti,Bakery,"Crunchy and delightful, this chocolate chip bi...","[Flour, Sugar, Chocolate Chips, Eggs, Almonds,...",2.50,4.6,chocolat_biscotti.jpg
4,Espresso shot,Coffee,"A bold shot of rich espresso, our espresso is ...",[Espresso],2.00,4.9,Espresso_shot.webp
5,Hazelnut Biscotti,Bakery,These delicious hazelnut biscotti are perfect ...,"[Flour, Sugar, Hazelnuts, Eggs, Baking Powder]",2.75,4.4,Hazelnut_Biscotti.jpg
6,Chocolate Croissant,Bakery,"Flaky and buttery, our chocolate croissant is ...","[Flour, Butter, Chocolate, Yeast, Sugar, Salt]",3.75,4.8,Chocolate_Croissant.jpg
7,Dark chocolate,Drinking Chocolate,"Rich and indulgent, our dark chocolate drinkin...","[Cocoa Powder, Sugar, Milk]",5.00,4.7,Dark_chocolate.jpg
8,Cranberry Scone,Bakery,This delightful cranberry scone combines sweet...,"[Flour, Butter, Cranberries, Sugar, Baking Pow...",3.50,4.5,Cranberry_Scone.jpg
9,Croissant,Bakery,"Our classic croissant is flaky and buttery, of...","[Flour, Butter, Yeast, Sugar, Salt]",3.25,4.7,Croissant.jpg


In [8]:
df['Text'] = df["name"] + " : " + df['description'] + " --Ingredients: " + df["ingredients"].astype(str) + " --Price: " + df["price"].astype(str) + " --rating: " + df["rating"].astype(str)

In [9]:
df['Text'][0]

"Cappuccino : A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations. --Ingredients: ['Espresso', 'Steamed Milk', 'Milk Foam'] --Price: 4.5 --rating: 4.7"

In [13]:
texts  = df['Text'].tolist()
texts

["Cappuccino : A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations. --Ingredients: ['Espresso', 'Steamed Milk', 'Milk Foam'] --Price: 4.5 --rating: 4.7",
 "Jumbo Savory Scone : Deliciously flaky and buttery, this jumbo savory scone is filled with herbs and cheese, creating a mouthwatering experience. Perfect for a hearty snack or a light lunch, it pairs beautifully with your favorite coffee or tea. --Ingredients: ['Flour', 'Butter', 'Cheese', 'Herbs', 'Baking Powder', 'Salt'] --Price: 3.25 --rating: 4.3",
 "Latte : Smooth and creamy, our latte combines rich espresso with velvety steamed milk, creating a perfect balance of flavor and texture. Enjoy it as a comforting treat any time of day, whether you're starting your morning or taking a midday break. --Ingredients: ['Espresso', '

Generating the embeddings

In [12]:
from openai import OpenAI
client = OpenAI(base_url ="http://localhost:12434/engines/llama.cpp/v1/", api_key = "anything")

In [18]:
output = client.embeddings.create(input = texts ,model="ai/embeddinggemma:latest")
embeddings = []
for embedding_object in output.data:
    embeddings.append(embedding_object.embedding)


In [19]:
embeddings

[[-0.010805883444845676,
  0.019755899906158447,
  -0.021185429766774178,
  0.02302876114845276,
  -0.03383183479309082,
  -0.012351509183645248,
  0.0023649055510759354,
  -0.022582748904824257,
  -0.018108131363987923,
  0.011614315211772919,
  -0.02804763801395893,
  0.03780115023255348,
  0.008695991709828377,
  -0.03100350871682167,
  -0.007580534089356661,
  -0.023986300453543663,
  -0.0362921841442585,
  0.004545201081782579,
  0.02142414264380932,
  -0.03072270192205906,
  -0.0007890920387580991,
  -0.0034468956291675568,
  0.022659974172711372,
  -0.019952455535531044,
  -0.05986516550183296,
  0.054157648235559464,
  0.0008467136649414897,
  -8.136285759974271e-05,
  0.006783023476600647,
  0.01667705923318863,
  -0.0163737665861845,
  -0.025338830426335335,
  -0.03289315849542618,
  -0.035616908222436905,
  -0.028652001172304153,
  -0.013590316288173199,
  -0.0013215771177783608,
  -0.010168492794036865,
  0.028530485928058624,
  -0.12870101630687714,
  0.031015992164611816,

In [21]:
len(embeddings[0])

768

CREATING THE DATA TO ADD IN THE JSON FILE

In [37]:
data = []
for index, text in enumerate(texts):
    sub_doc = {}
    sub_doc["text"] = text
    sub_doc["embeddings"] = embeddings[index]
    data.append(sub_doc)


print(len(data), data[0])

18 {'text': "Cappuccino : A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations. --Ingredients: ['Espresso', 'Steamed Milk', 'Milk Foam'] --Price: 4.5 --rating: 4.7", 'embeddings': [-0.010805883444845676, 0.019755899906158447, -0.021185429766774178, 0.02302876114845276, -0.03383183479309082, -0.012351509183645248, 0.0023649055510759354, -0.022582748904824257, -0.018108131363987923, 0.011614315211772919, -0.02804763801395893, 0.03780115023255348, 0.008695991709828377, -0.03100350871682167, -0.007580534089356661, -0.023986300453543663, -0.0362921841442585, 0.004545201081782579, 0.02142414264380932, -0.03072270192205906, -0.0007890920387580991, -0.0034468956291675568, 0.022659974172711372, -0.019952455535531044, -0.05986516550183296, 0.054157648235559464, 0.0008467136649414897, -8.136

In [38]:
import json
with open("embeddings.json", "w") as f:
    json.dump(data, f)

In [41]:
with open("embeddings.json", "r") as f:
    data = json.load(f)

    loaded_embeddings = [item["embeddings"] for item in data]

In [42]:
loaded_embeddings

[[-0.010805883444845676,
  0.019755899906158447,
  -0.021185429766774178,
  0.02302876114845276,
  -0.03383183479309082,
  -0.012351509183645248,
  0.0023649055510759354,
  -0.022582748904824257,
  -0.018108131363987923,
  0.011614315211772919,
  -0.02804763801395893,
  0.03780115023255348,
  0.008695991709828377,
  -0.03100350871682167,
  -0.007580534089356661,
  -0.023986300453543663,
  -0.0362921841442585,
  0.004545201081782579,
  0.02142414264380932,
  -0.03072270192205906,
  -0.0007890920387580991,
  -0.0034468956291675568,
  0.022659974172711372,
  -0.019952455535531044,
  -0.05986516550183296,
  0.054157648235559464,
  0.0008467136649414897,
  -8.136285759974271e-05,
  0.006783023476600647,
  0.01667705923318863,
  -0.0163737665861845,
  -0.025338830426335335,
  -0.03289315849542618,
  -0.035616908222436905,
  -0.028652001172304153,
  -0.013590316288173199,
  -0.0013215771177783608,
  -0.010168492794036865,
  0.028530485928058624,
  -0.12870101630687714,
  0.031015992164611816,

In [43]:
from sklearn.metrics.pairwise import cosine_similarity

In [44]:
user_prompt_embedding = client.embeddings.create(input="is cappuccino avaiable?", model= "ai/embeddinggemma:latest")
user_prompt_embedding

CreateEmbeddingResponse(data=[Embedding(embedding=[-0.02669430710375309, 0.03712988272309303, -0.0296404417604208, 0.04395923390984535, -0.0030889874324202538, -0.012773401103913784, 0.02376328408718109, -0.0102853337302804, 0.0008422191021963954, 0.00333508662879467, -0.012023771181702614, 0.053583476692438126, 0.04720139130949974, -0.059686169028282166, -0.016142316162586212, -0.002245647832751274, -0.021627584472298622, 0.02036774531006813, 0.05203171446919441, -0.04506824538111687, 0.003778760088607669, 0.010889940895140171, 0.003024559235200286, -0.017502855509519577, -0.005231874063611031, 0.04325408115983009, -0.0037424422334879637, 0.01119361724704504, -0.022467194125056267, -0.007143329828977585, -0.015438918024301529, -0.010430995374917984, -0.02299482934176922, -0.025115901604294777, -0.02576950564980507, -0.013118386268615723, -0.006529805716127157, -0.009264422580599785, 0.026656094938516617, -0.30059653520584106, -0.06784630566835403, -0.01643107458949089, 0.0827971994876

In [45]:
user_prompt_embedding = user_prompt_embedding.data[0].embedding
user_prompt_embedding

[-0.02669430710375309,
 0.03712988272309303,
 -0.0296404417604208,
 0.04395923390984535,
 -0.0030889874324202538,
 -0.012773401103913784,
 0.02376328408718109,
 -0.0102853337302804,
 0.0008422191021963954,
 0.00333508662879467,
 -0.012023771181702614,
 0.053583476692438126,
 0.04720139130949974,
 -0.059686169028282166,
 -0.016142316162586212,
 -0.002245647832751274,
 -0.021627584472298622,
 0.02036774531006813,
 0.05203171446919441,
 -0.04506824538111687,
 0.003778760088607669,
 0.010889940895140171,
 0.003024559235200286,
 -0.017502855509519577,
 -0.005231874063611031,
 0.04325408115983009,
 -0.0037424422334879637,
 0.01119361724704504,
 -0.022467194125056267,
 -0.007143329828977585,
 -0.015438918024301529,
 -0.010430995374917984,
 -0.02299482934176922,
 -0.025115901604294777,
 -0.02576950564980507,
 -0.013118386268615723,
 -0.006529805716127157,
 -0.009264422580599785,
 0.026656094938516617,
 -0.30059653520584106,
 -0.06784630566835403,
 -0.01643107458949089,
 0.08279719948768616,
 0

In [46]:
similarity_scores = cosine_similarity([user_prompt_embedding], loaded_embeddings)
similarity_scores

array([[0.60095405, 0.41016728, 0.50936187, 0.4506582 , 0.51603465,
        0.46166458, 0.42678918, 0.41816647, 0.41471215, 0.40839438,
        0.39525455, 0.39954331, 0.40426727, 0.45035775, 0.4325    ,
        0.44422823, 0.46710643, 0.41465884]])

In [48]:
closest_entry = similarity_scores.argmax()
closest_text = data[closest_entry]["text"]
closest_text

"Cappuccino : A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations. --Ingredients: ['Espresso', 'Steamed Milk', 'Milk Foam'] --Price: 4.5 --rating: 4.7"

In [51]:
user_prompt = f"""
{closest_text}
"is cappucino avaiable"
"""

In [52]:
print(user_prompt)
input  = [{"role":"user" , "content":user_prompt}]
output = client.chat.completions.create(
    model= "ai/llama3.2:1B-Q4_0",
    messages=input
)
print(output.choices[0].message.content)


Cappuccino : A rich and creamy cappuccino made with freshly brewed espresso, steamed milk, and a frothy milk cap. This delightful drink offers a perfect balance of bold coffee flavor and smooth milk, making it an ideal companion for relaxing mornings or lively conversations. --Ingredients: ['Espresso', 'Steamed Milk', 'Milk Foam'] --Price: 4.5 --rating: 4.7
"is cappucino avaiable"

The question "Is cappuccino available?" can be analyzed using the given information:

- Ingredients: ['Espresso', 'Steamed Milk', 'Milk Foam']
- Price: 4.5
- Rating: 4.7

Given the ingredients and price of the cappuccino, the question can be answered as follows:

Yes, cappuccino is available. The ingredients and price are within the range of what is typically available for the cappuccino.
